# **Tugas Besar Big Data : E-Commerce Olist Brazil**

## **Bagian 1. Pipline ETL (Extract - Transform - Load)**

### **1.1 EXTRACT**

Data diambil dari **dua sumber berbeda**:
- **Sumber 1:** File CSV hasil gabungan 8 dataset Olist (diunggah ke Google Drive)
- **Sumber 2:** File JSON kurs historis BRL/USD (representasi response dari Frankfurter API)

In [78]:
import pandas as pd
import os
import time
import json
import urllib.request

# 1. ID File dari Google Drive
ID_FILE_CSV  = '1ju7BogZ9kOrFFSSanZZt7pgxYh13aemZ'
ID_FILE_JSON = '1_ciD4VwgfG4HJezwFvaq5VIMwv0O-Llj'

extraction_logs = []

def generate_log(source_name, df, start_time):
    execution_time = time.time() - start_time
    return {
        "nama sumber data"      : source_name,
        "jumlah baris"          : df.shape[0],
        "jumlah kolom"          : df.shape[1],
        "waktu eksekusi (detik)": round(execution_time, 4)
    }

# --- SUMBER 1 (CSV) ---
def extract_etl_source1():
    start_time = time.time()
    source_name = "Olist_Ecommerce_CSV_GDrive"
    url = f'https://drive.google.com/uc?export=download&id={ID_FILE_CSV}'
    df_raw = pd.read_csv(url, low_memory=False)
    log = generate_log(source_name, df_raw, start_time)
    extraction_logs.append(log)
    print(f"{source_name}: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
    return df_raw

# --- SUMBER 2 (JSON) ---
def extract_etl_source2():
    start_time = time.time()
    source_name = "Frankfurter_API_BRL_USD_Hourly_JSON"
    url = f'https://drive.google.com/uc?export=download&id={ID_FILE_JSON}'
    urllib.request.urlretrieve(url, 'brl_usd_hourly.json')
    with open('brl_usd_hourly.json') as f:
        raw = json.load(f)
    # Handle berbagai struktur JSON
    if isinstance(raw, list):
        df_raw = pd.DataFrame(raw)
    elif isinstance(raw, dict):
        # Cari key yang isinya list of records
        data_key = next((k for k in raw if isinstance(raw[k], list)), None)
        if data_key:
            df_raw = pd.DataFrame(raw[data_key])
        else:
            df_raw = pd.DataFrame([raw])
    log = generate_log(source_name, df_raw, start_time)
    extraction_logs.append(log)
    print(f"{source_name}: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
    return df_raw, raw

# --- EKSEKUSI ---
print("Memulai Tahap Extract...")
try:
    df_orders_raw = extract_etl_source1()
    df_rates_raw, rates_meta = extract_etl_source2()

    print("\n" + "="*75)
    print("LOG PROSES EXTRACT (DATA MENTAH)")
    print("="*75)
    display(pd.DataFrame(extraction_logs))
    print("="*75)
except Exception as e:
    print(f"Error: {e}")

Memulai Tahap Extract...
Olist_Ecommerce_CSV_GDrive: 113425 baris, 42 kolom
Frankfurter_API_BRL_USD_Hourly_JSON: 35064 baris, 11 kolom

LOG PROSES EXTRACT (DATA MENTAH)


,nama sumber data,jumlah baris,jumlah kolom,waktu eksekusi (detik)
0,Olist_Ecommerce_CSV_GDrive,113425,42,6.2234
1,Frankfurter_API_BRL_USD_Hourly_JSON,35064,11,1.8402


### **1.2 TRANSFORM**

#### A. Data Cleaning


In [79]:
import pandas as pd
import numpy as np
import json

print("Memulai Tahap Transform...")

# 1. LOAD DATA DARI GOOGLE DRIVE
df_orders = df_orders_raw.copy()

# Load JSON kurs — handle berbagai struktur
with open('brl_usd_hourly.json') as f:
    raw_json = json.load(f)

if isinstance(raw_json, list):
    df_rates = pd.DataFrame(raw_json)
elif isinstance(raw_json, dict):
    data_key = next((k for k in raw_json if isinstance(raw_json[k], list)), None)
    if data_key:
        df_rates = pd.DataFrame(raw_json[data_key])
    else:
        df_rates = pd.DataFrame([raw_json])

print(f"Kolom JSON: {list(df_rates.columns)}")

# 2. JOIN DATA — Gabungkan berdasarkan tanggal order dengan kurs harian
df_orders['order_date'] = pd.to_datetime(df_orders['order_purchase_timestamp']).dt.strftime('%Y-%m-%d')

# Cari kolom tanggal di JSON
date_col  = next((c for c in df_rates.columns if 'date' in c.lower()), df_rates.columns[0])
rate_col  = next((c for c in df_rates.columns if 'close' in c.lower()), None)
vol_col   = next((c for c in df_rates.columns if 'volume' in c.lower()), None)

agg_dict = {}
if rate_col:  agg_dict[rate_col] = 'mean'
if vol_col:   agg_dict[vol_col]  = 'mean'

if agg_dict:
    df_rates_daily = df_rates.groupby(date_col).agg(agg_dict).reset_index()
    df_rates_daily = df_rates_daily.rename(columns={
        date_col: 'order_date',
        rate_col: 'close_rate'
    })
else:
    df_rates_daily = df_rates.rename(columns={date_col: 'order_date'})

df_transformed = pd.merge(df_orders, df_rates_daily, on='order_date', how='left')

print(f"Dimensi awal setelah Join: {df_transformed.shape[0]} baris, {df_transformed.shape[1]} kolom")


Memulai Tahap Transform...
Kolom JSON: ['date', 'hour', 'datetime', 'base_currency', 'target_currency', 'open_rate', 'high_rate', 'low_rate', 'close_rate', 'volume_brl', 'source']
Dimensi awal setelah Join: 113425 baris, 45 kolom


In [80]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##### 1. Menghapus Duplicate Data

In [81]:
print(f"Jumlah baris sebelum drop duplicates: {len(df_transformed)}")

# Menghapus duplicate
df_transformed = df_transformed.drop_duplicates()

print(f"Jumlah baris setelah drop duplicates: {len(df_transformed)}")
df_transformed.head()

Jumlah baris sebelum drop duplicates: 113425
Jumlah baris setelah drop duplicates: 113425


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_lat,customer_lng,seller_lat,seller_lng,year_month,brl_to_usd_rate,payment_value_usd,order_date,close_rate,volume_brl
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,-23.576983,-46.587161,-23.680729,-46.444238,2017-10,0.3150,12.19,2017-10-02,0.314841,3.006665e+07
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,-12.177924,-44.660711,-19.807681,-43.980427,2018-07,0.2599,36.77,2018-07-24,0.259683,2.699925e+07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,-16.745150,-48.514783,-21.363502,-48.229601,2018-08,0.2484,44.49,2018-08-08,0.248420,2.680703e+07
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,-5.774190,-35.271143,-19.837682,-43.924053,2017-11,0.3055,22.06,2017-11-18,0.305689,2.441279e+07
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,-23.676370,-46.514627,-23.543395,-46.262086,2018-02,0.3045,8.71,2018-02-13,0.304634,2.997216e+07


##### 2. Menangani Missing Value

In [82]:
# Cek jumlah missing value sebelum ditangani
print("Jumlah Missing Value SEBELUM:")
print(df_transformed.isnull().sum())

# Proses Pengisian (Median untuk angka, 'Unknown' untuk teks)
for col in df_transformed.columns:
    if df_transformed[col].dtype == 'object':
        df_transformed[col] = df_transformed[col].fillna('Unknown')
    else:
        df_transformed[col] = df_transformed[col].fillna(df_transformed[col].median())

print("\nJumlah Missing Value SESUDAH:")
print(df_transformed.isnull().sum())
df_transformed.head()

Jumlah Missing Value SEBELUM:
order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                  161
order_delivered_carrier_date      1968
order_delivered_customer_date     3229
order_estimated_delivery_date        0
customer_unique_id                   0
customer_zip_code_prefix             0
customer_city                        0
customer_state                       0
order_item_id                      775
product_id                         775
seller_id                          775
shipping_limit_date                775
price                              775
freight_value                      775
product_category_name             2378
product_name_lenght               2378
product_description_lenght        2378
product_photos_qty                2378
product_weight_g                   793
product_length_cm                  793
product_height_cm                 

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_lat,customer_lng,seller_lat,seller_lng,year_month,brl_to_usd_rate,payment_value_usd,order_date,close_rate,volume_brl
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,-23.576983,-46.587161,-23.680729,-46.444238,2017-10,0.3150,12.19,2017-10-02,0.314841,3.006665e+07
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,-12.177924,-44.660711,-19.807681,-43.980427,2018-07,0.2599,36.77,2018-07-24,0.259683,2.699925e+07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,-16.745150,-48.514783,-21.363502,-48.229601,2018-08,0.2484,44.49,2018-08-08,0.248420,2.680703e+07
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,-5.774190,-35.271143,-19.837682,-43.924053,2017-11,0.3055,22.06,2017-11-18,0.305689,2.441279e+07
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,-23.676370,-46.514627,-23.543395,-46.262086,2018-02,0.3045,8.71,2018-02-13,0.304634,2.997216e+07


##### 3. Standarisasi Format Tanggal

In [83]:
# Mengubah ke format datetime standar
df_transformed['shipping_limit_date'] = pd.to_datetime(
    df_transformed['shipping_limit_date'],
    errors='coerce'
)

print("Tipe data kolom tanggal sekarang:")
print(df_transformed['shipping_limit_date'].dtype)

df_transformed[['shipping_limit_date']].head()

Tipe data kolom tanggal sekarang:
datetime64[ns]


,shipping_limit_date
0,2017-10-06 11:07:15
1,2018-07-30 03:24:27
2,2018-08-13 08:55:23
3,2017-11-23 19:45:59
4,2018-02-19 20:31:37


##### 4. Menangani Outlier — Metode IQR (price & freight_value)

In [84]:
# Hitung Batas IQR
Q1 = df_transformed['price'].quantile(0.25)
Q3 = df_transformed['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Proses Capping (Membatasi nilai)
df_transformed['price'] = df_transformed['price'].clip(lower=lower_bound, upper=upper_bound)

print(f"Batas Bawah: {lower_bound}, Batas Atas: {upper_bound}")
print("Data Harga (Price) setelah penanganan outlier:")
df_transformed[['price']].head()

Batas Bawah: -101.24999999999997, Batas Atas: 275.15
Data Harga (Price) setelah penanganan outlier:


,price
0,29.99
1,118.70
2,159.90
3,45.00
4,19.90


#### B. Standarisasi Data


##### 1. Menyeragamkan Penamaan Kolom (lowercase & snake_case)

In [85]:
# Menyeragamkan nama kolom: Huruf kecil dan ganti spasi dengan underscore
df_transformed.columns = [col.lower().replace(' ', '_').strip() for col in df_transformed.columns]

print("Daftar nama kolom setelah standardisasi:")
print(df_transformed.columns.tolist())
df_transformed.head(2)

Daftar nama kolom setelah standardisasi:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_type', 'payment_installments', 'payment_value', 'review_score', 'review_comment_title', 'review_comment_message', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'year_month', 'brl_to_usd_rate', 'payment_value_usd', 'order_date', 'close_rate', 'volume_brl']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_lat,customer_lng,seller_lat,seller_lng,year_month,brl_to_usd_rate,payment_value_usd,order_date,close_rate,volume_brl
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,-23.576983,-46.587161,-23.680729,-46.444238,2017-10,0.3150,12.19,2017-10-02,0.314841,3.006665e+07
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,-12.177924,-44.660711,-19.807681,-43.980427,2018-07,0.2599,36.77,2018-07-24,0.259683,2.699925e+07


##### 2. Normalisasi Pada Kolom Numerik

In [86]:
# Normalisasi menggunakan Min-Max Scaling
for col in ['price', 'freight_value']:
    min_val = df_transformed[col].min()
    max_val = df_transformed[col].max()
    df_transformed[f'{col}_normalized'] = (df_transformed[col] - min_val) / (max_val - min_val)

print("Hasil normalisasi kolom 'price' dan 'freight_value':")
df_transformed[['price', 'price_normalized', 'freight_value', 'freight_value_normalized']].head()


Hasil normalisasi kolom 'price' dan 'freight_value':


,price,price_normalized,freight_value,freight_value_normalized
0,29.99,0.106234,8.72,0.021285
1,118.70,0.429639,22.76,0.055556
2,159.90,0.579840,19.22,0.046915
3,45.00,0.160955,27.20,0.066393
4,19.90,0.069450,8.72,0.021285


##### 3. Encoding Kolom Kategorikal

In [87]:
# Melakukan Encoding (Label Encoding sederhana)
df_transformed['category_encoded'] = df_transformed['product_category_name'].astype('category').cat.codes

print("Hasil encoding pada kolom kategori produk:")
df_transformed[['product_category_name', 'category_encoded']].head()


Hasil encoding pada kolom kategori produk:


,product_category_name,category_encoded
0,utilidades_domesticas,73
1,perfumaria,63
2,automotivo,9
3,pet_shop,64
4,papelaria,60


##### 4. Konsistensi Tipe Data

In [88]:
# 1. Identifikasi kolom numerik yang akan diproses
numeric_cols = ['price', 'freight_value', 'product_weight_g']

# 2. Tangani string kosong ('') yang menyebabkan error
for col in numeric_cols:
    if col in df_transformed.columns:
        # Ubah string kosong menjadi NaN agar bisa diproses sebagai angka
        df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
        # Isi kembali nilai NaN tersebut dengan median agar tidak ada data kosong
        df_transformed[col] = df_transformed[col].fillna(df_transformed[col].median())

# 3. Definisikan konfigurasi tipe data akhir
data_types_config = {
    'order_id'        : 'object',
    'product_id'      : 'object',
    'seller_id'       : 'object',
    'price'           : 'float64',
    'freight_value'   : 'float64',
    'product_weight_g': 'float64',
    'close_rate'      : 'float64',
}

# 4. Terapkan perubahan tipe data dengan aman
for col, dtype in data_types_config.items():
    if col in df_transformed.columns:
        df_transformed[col] = df_transformed[col].astype(dtype)

print("Tipe data setelah standardisasi:")
print(df_transformed[list(data_types_config.keys())].dtypes)


Tipe data setelah standardisasi:
order_id             object
product_id           object
seller_id            object
price               float64
freight_value       float64
product_weight_g    float64
close_rate          float64
dtype: object


###### Penjelasan

1. Lowercase & Snake Case: Memudahkan integrasi dengan database SQL yang umumnya case-insensitive namun lebih nyaman dengan format snake_case.

2. Normalisasi: Penting jika data ini nantinya digunakan untuk Machine Learning atau analisis statistik agar kolom dengan skala besar (misal: harga ribuan) tidak mendominasi kolom dengan skala kecil (misal: berat).

3. Encoding: Mengubah teks menjadi representasi numerik adalah standar dalam pemrosesan data besar (Big Data) untuk efisiensi penyimpanan dan komputasi.

#### C. Data Enrichment dan Feature Engineering



##### Data Aggregation


###### Agregasi 1 : Harga & Ongkir per Kategori Produk

Groupby 1 : product_category_name
Metrics: mean, median, count

In [89]:
# AGREGASI 1: Statistik harga dan ongkir per kategori produk
agg_category = (
    df_transformed
    .groupby('product_category_name')
    .agg(
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        total_orders=('order_id', 'count'),
        avg_freight=('freight_value', 'mean'),
        median_freight=('freight_value', 'median')
    )
    .reset_index()
)

print("Agregasi berdasarkan kategori produk:")
display(agg_category.head())

Agregasi berdasarkan kategori produk:


,product_category_name,avg_price,median_price,total_orders,avg_freight,median_freight
0,Unknown,87.039197,74.990,2378,17.145210,16.26
1,agro_industria_e_comercio,170.542075,258.650,212,27.564151,23.68
2,alimentos,57.634137,48.945,510,14.256922,12.94
3,alimentos_bebidas,52.013957,38.745,278,16.215791,16.11
4,artes,85.904067,97.500,209,19.354880,15.23


###### Penjelasan:

Mengelompokkan data berdasarkan kategori produk

*  Mengelompokkan data berdasarkan kategori produk
*   Mean dan median digunakan untuk melihat kecenderungan harga pada setiap kategori
* Count digunakan untuk mengetahui jumlah transaksi pada tiap kategori
* Membantu mengidentifikasi kategori dengan volume dan harga transaksi tertinggi

###### Agregasi 2 : Bulan Pembelian

Groupby: purchase_month
Metrik: mean, median, count

In [90]:
# AGREGASI 2: Statistik transaksi per bulan pembelian
df_transformed['purchase_month'] = pd.to_datetime(df_transformed['order_purchase_timestamp']).dt.month

agg_monthly = (
    df_transformed
    .groupby('purchase_month')
    .agg(
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        total_orders=('order_id', 'count'),
        avg_close_rate=('close_rate', 'mean')
    )
    .reset_index()
    .sort_values('purchase_month')
)

print("Agregasi berdasarkan bulan pembelian:")
display(agg_monthly)


Agregasi berdasarkan bulan pembelian:


,purchase_month,avg_price,median_price,total_orders,avg_close_rate
0,1,97.703567,75.00,9223,0.308885
1,2,94.935614,72.99,9704,0.307934
2,3,98.357751,74.99,11281,0.304736
3,4,100.899813,79.00,10677,0.299485
4,5,99.237950,77.98,12121,0.286068
5,6,97.574870,75.00,10696,0.275733
6,7,98.338488,74.99,11687,0.281307
7,8,95.955308,69.90,12256,0.275989
8,9,101.541853,74.99,4896,0.316183
9,10,99.587790,74.90,5768,0.313821


###### Penjelasan:

*   Mengelompokkan produk berdasarkan segmen harga Low, Medium, dan High
*   Mean dan median menunjukkan karakteristik harga pada tiap segmen
* Count menunjukkan segmen harga dengan jumlah transaksi terbanyak
* Mendukung analisis strategi penetapan harga produk

##### **1. Data Enrichment (Join/Merge)**

Penggabungan antara data transaksi (orders) dan data kurs harian BRL/USD.

In [91]:
# Data Enrichment: Konversi nilai pembayaran ke USD menggunakan close_rate harian
df_transformed['payment_value_usd'] = (
    df_transformed['payment_value'] * df_transformed['close_rate']
).round(2)

print(f"Hasil Enrichment memiliki {df_transformed.shape[1]} kolom.")
df_transformed[['order_id', 'payment_value', 'close_rate', 'payment_value_usd']].head(2)


Hasil Enrichment memiliki 49 kolom.


,order_id,payment_value,close_rate,payment_value_usd
0,e481f51cbdc54678b7cc49136f2d6af7,38.71,0.314841,12.19
1,53cdb2fc8bc7dce0b6741e2150273451,141.46,0.259683,36.73


##### **2. Feature Engineering (Membuat 5 Fitur Baru)**

Membuat fitur berbasis waktu, rasio, dan kategori.

In [92]:
# 1. Fitur Berbasis Waktu: 'shipping_limit_month'
df_transformed['shipping_limit_date'] = pd.to_datetime(df_transformed['shipping_limit_date'], errors='coerce')
df_transformed['shipping_limit_month'] = df_transformed['shipping_limit_date'].dt.month

# 2. Fitur Rasio: 'freight_ratio'
df_transformed['freight_ratio'] = df_transformed['freight_value'] / df_transformed['price'].replace(0, 0.01)

# 3. Fitur Dimensi: 'product_volume_cm3'
for col in ['product_length_cm', 'product_height_cm', 'product_width_cm']:
    df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
    df_transformed[col] = df_transformed[col].fillna(df_transformed[col].median())

df_transformed['product_volume_cm3'] = (
    df_transformed['product_length_cm'] *
    df_transformed['product_height_cm'] *
    df_transformed['product_width_cm']
)

# 4. Segmen Harga: 'price_segment'
df_transformed['price_segment'] = pd.cut(
    df_transformed['price'],
    bins=[0, 50, 200, float('inf')],
    labels=['Low', 'Medium', 'High']
)

# 5. Kurs Tier: 'rate_tier' (dari data JSON kurs harian)
df_transformed['rate_tier'] = pd.cut(
    df_transformed['close_rate'],
    bins=[0, 0.27, 0.30, float('inf')],
    labels=['Weak BRL', 'Normal BRL', 'Strong BRL']
)

print("5 Fitur baru berhasil dibuat:")
print(df_transformed[['shipping_limit_month', 'freight_ratio',
                       'product_volume_cm3', 'price_segment', 'rate_tier']].head())


5 Fitur baru berhasil dibuat:
   shipping_limit_month  freight_ratio  product_volume_cm3 price_segment  \
0                  10.0       0.290764              1976.0           Low   
1                   7.0       0.191744              4693.0        Medium   
2                   8.0       0.120200              9576.0        Medium   
3                  11.0       0.604444              6000.0           Low   
4                   2.0       0.438191             11475.0           Low   

    rate_tier  
0  Strong BRL  
1    Weak BRL  
2    Weak BRL  
3  Strong BRL  
4  Strong BRL  


In [93]:
# Menampilkan informasi ringkas tentang dataframe final
print("="*80)
print("RINGKASAN DATAFRAME FINAL (ETL RESULT)")
print("="*80)
print(f"Total Baris  : {df_transformed.shape[0]}")
print(f"Total Kolom  : {df_transformed.shape[1]}")
print("-"*80)

display(df_transformed.head(5))

print("\nDAFTAR KOLOM YANG TERSEDIA:")
for i, col in enumerate(df_transformed.columns, 1):
    print(f"{i}. {col}")


RINGKASAN DATAFRAME FINAL (ETL RESULT)
Total Baris  : 113425
Total Kolom  : 54
--------------------------------------------------------------------------------


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,volume_brl,price_normalized,freight_value_normalized,category_encoded,purchase_month,shipping_limit_month,freight_ratio,product_volume_cm3,price_segment,rate_tier
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.006665e+07,0.106234,0.021285,73,10,10.0,0.290764,1976.0,Low,Strong BRL
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,2.699925e+07,0.429639,0.055556,63,7,7.0,0.191744,4693.0,Medium,Weak BRL
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,2.680703e+07,0.579840,0.046915,9,8,8.0,0.120200,9576.0,Medium,Weak BRL
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,2.441279e+07,0.160955,0.066393,64,11,11.0,0.604444,6000.0,Low,Strong BRL
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,2.997216e+07,0.069450,0.021285,60,2,2.0,0.438191,11475.0,Low,Strong BRL



DAFTAR KOLOM YANG TERSEDIA:
1. order_id
2. customer_id
3. order_status
4. order_purchase_timestamp
5. order_approved_at
6. order_delivered_carrier_date
7. order_delivered_customer_date
8. order_estimated_delivery_date
9. customer_unique_id
10. customer_zip_code_prefix
11. customer_city
12. customer_state
13. order_item_id
14. product_id
15. seller_id
16. shipping_limit_date
17. price
18. freight_value
19. product_category_name
20. product_name_lenght
21. product_description_lenght
22. product_photos_qty
23. product_weight_g
24. product_length_cm
25. product_height_cm
26. product_width_cm
27. seller_zip_code_prefix
28. seller_city
29. seller_state
30. payment_type
31. payment_installments
32. payment_value
33. review_score
34. review_comment_title
35. review_comment_message
36. customer_lat
37. customer_lng
38. seller_lat
39. seller_lng
40. year_month
41. brl_to_usd_rate
42. payment_value_usd
43. order_date
44. close_rate
45. volume_brl
46. price_normalized
47. freight_value_normalized

###### Analisis
1. Integrasi Data: Kolom seperti close_rate dan payment_value_usd membuktikan penggabungan sumber CSV dan JSON kurs harian berhasil.

2. Standardisasi: Semua nama kolom sudah menggunakan format lowercase dan snake_case (contoh: shipping_limit_date, freight_value).

3. Feature Engineering: Adanya kolom baru seperti freight_ratio, product_volume_cm3, price_segment, dan rate_tier menunjukkan proses pengayaan data.

4. Normalisasi & Encoding: Kolom price_normalized (rentang 0-1) dan category_encoded (angka) membuktikan tahap transformasi lanjutan telah dilakukan.

#### D. Validasi Kualitas Data

##### **1.  Inisialisasi Laporan Validasi**

Jalankan ini untuk menyiapkan tabel rangkuman hasil validasi.

In [94]:
# List untuk menampung hasil validasi
validation_logs = []

def record_validation(test_name, status, detail):
    validation_logs.append({
        "Aturan Validasi": test_name,
        "Status": status,
        "Keterangan": detail
    })

print("Logger validasi siap.")

Logger validasi siap.


##### **2. Uniqueness & Null Check**

Mengecek duplikasi pada Primary Key dan memastikan tidak ada data yang kosong.

In [95]:
# 1. Uniqueness Check
dup_pk = df_transformed.duplicated(subset=['order_id', 'product_id']).sum()
if dup_pk == 0:
    record_validation("Uniqueness Check", "LULUS", "Tidak ada duplikat pada Primary Key.")
else:
    df_transformed = df_transformed.drop_duplicates(subset=['order_id', 'product_id'])
    record_validation("Uniqueness Check", "DIPERBAIKI", f"{dup_pk} baris duplikat dihapus.")

# 2. Null Check
null_count = df_transformed.isnull().sum().sum()
if null_count == 0:
    record_validation("Null Check", "LULUS", "Semua kolom terisi (0 null).")
else:
    record_validation("Null Check", "GAGAL", f"Terdeteksi {null_count} data kosong.")

print("Hasil Uniqueness & Null Check:")
display(df_transformed.isnull().sum().to_frame(name='jumlah_null'))


Hasil Uniqueness & Null Check:


,jumlah_null
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,0
order_delivered_carrier_date,0
order_delivered_customer_date,0
order_estimated_delivery_date,0
customer_unique_id,0
customer_zip_code_prefix,0


##### **3. Range & Datatype Check**

Memastikan nilai numerik masuk akal (tidak negatif) dan tipe data sudah benar.

In [96]:
# 3. Range Check (Harga tidak boleh <= 0)
invalid_price = df_transformed[df_transformed['price'] <= 0].shape[0]
if invalid_price == 0:
    record_validation("Range Check", "LULUS", "Harga berada dalam rentang positif.")
else:
    df_transformed = df_transformed[df_transformed['price'] > 0]
    record_validation("Range Check", "DIPERBAIKI", f"{invalid_price} data harga tidak valid dihapus.")

# 4. Datatype Consistency
if df_transformed['price'].dtype == 'float64':
    record_validation("Datatype Consistency", "LULUS", "Tipe data kolom 'price' sudah float64.")
else:
    record_validation("Datatype Consistency", "GAGAL", "Tipe data tidak sesuai.")

print(f"Batas harga minimum saat ini: {df_transformed['price'].min()}")


Batas harga minimum saat ini: 0.85


##### **4. Referential Integrity & Distribusi Data**

Memastikan hubungan antar tabel valid dan data memiliki variasi nilai.

In [97]:
# 5. Referential Integrity
ref_issue = df_transformed[df_transformed['product_id'] == 'Unknown'].shape[0]
if ref_issue == 0:
    record_validation("Referential Integrity", "LULUS", "Integritas referensi product_id terjaga.")
else:
    record_validation("Referential Integrity", "GAGAL", f"Ada {ref_issue} data tanpa Product ID.")

# 6. Distribusi Data (Pengecekan Variansi)
unique_segments = df_transformed['price_segment'].nunique()
if unique_segments > 1:
    record_validation("Distribusi Data", "LULUS", f"Data bervariasi ({unique_segments} kategori).")
else:
    record_validation("Distribusi Data", "GAGAL", "Data tidak bervariasi (hanya 1 kategori).")

print("Distribusi Kategori Harga:")
print(df_transformed['price_segment'].value_counts())


Distribusi Kategori Harga:
price_segment
Medium    55982
Low       34409
High      12809
Name: count, dtype: int64


##### **5. Laporan Kualitas Data Final**

Tampilkan tabel rangkuman untuk membuktikan semua aturan sudah diperiksa.

In [98]:
# Membuat DataFrame dari hasil log
df_val_report = pd.DataFrame(validation_logs)

print("="*80)
print("TABEL VALIDASI KUALITAS DATA AKHIR")
print("="*80)
display(df_val_report)
print("="*80)

# Tampilkan head data final
print("\nDATA FINAL SIAP LOAD:")
display(df_transformed.head())


TABEL VALIDASI KUALITAS DATA AKHIR


,Aturan Validasi,Status,Keterangan
0,Uniqueness Check,DIPERBAIKI,10225 baris duplikat dihapus.
1,Null Check,GAGAL,Terdeteksi 1550 data kosong.
2,Range Check,LULUS,Harga berada dalam rentang positif.
3,Datatype Consistency,LULUS,Tipe data kolom 'price' sudah float64.
4,Referential Integrity,GAGAL,Ada 775 data tanpa Product ID.
5,Distribusi Data,LULUS,Data bervariasi (3 kategori).



DATA FINAL SIAP LOAD:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,volume_brl,price_normalized,freight_value_normalized,category_encoded,purchase_month,shipping_limit_month,freight_ratio,product_volume_cm3,price_segment,rate_tier
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.006665e+07,0.106234,0.021285,73,10,10.0,0.290764,1976.0,Low,Strong BRL
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,2.699925e+07,0.429639,0.055556,63,7,7.0,0.191744,4693.0,Medium,Weak BRL
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,2.680703e+07,0.579840,0.046915,9,8,8.0,0.120200,9576.0,Medium,Weak BRL
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,2.441279e+07,0.160955,0.066393,64,11,11.0,0.604444,6000.0,Low,Strong BRL
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,2.997216e+07,0.069450,0.021285,60,2,2.0,0.438191,11475.0,Low,Strong BRL


### **1.3 LOAD**

##### **1. Desain Skema & Load Data**

Script ini akan membuat database, mendefinisikan skema dengan Primary Key dan Foreign Key, serta mencatat log eksekusi.

In [99]:
import sqlite3
import time

# 1. Inisialisasi Database dan Log
db_name = "olist_data_warehouse.db"
conn = sqlite3.connect(db_name)
cursor = conn.cursor()
load_logs = []

def record_load_log(step, status):
    load_logs.append({
        "Langkah": step,
        "Status": status,
        "Waktu Eksekusi": time.strftime("%Y-%m-%d %H:%M:%S")
    })

start_load = time.time()

try:
    # 2. Membuat Tabel Dimensi (dim_products)
    cursor.execute("DROP TABLE IF EXISTS dim_products")
    cursor.execute("""
        CREATE TABLE dim_products (
            product_id              TEXT PRIMARY KEY,
            product_category_name   TEXT,
            product_weight_g        REAL,
            product_volume_cm3      REAL,
            category_encoded        INTEGER
        )
    """)

    # 3. Membuat Tabel Dimensi (dim_exchange_rates) — dari sumber JSON
    cursor.execute("DROP TABLE IF EXISTS dim_exchange_rates")
    cursor.execute("""
        CREATE TABLE dim_exchange_rates (
            date            TEXT PRIMARY KEY,
            open_rate       REAL,
            high_rate       REAL,
            low_rate        REAL,
            close_rate      REAL,
            volume_brl      REAL,
            base_currency   TEXT,
            target_currency TEXT
        )
    """)

    # 4. Membuat Tabel Fakta (fact_orders)
    cursor.execute("DROP TABLE IF EXISTS fact_orders")
    cursor.execute("""
        CREATE TABLE fact_orders (
            order_id                TEXT,
            product_id              TEXT,
            price                   REAL,
            freight_value           REAL,
            freight_ratio           REAL,
            payment_value           REAL,
            payment_value_usd       REAL,
            close_rate              REAL,
            rate_tier               TEXT,
            price_segment           TEXT,
            order_status            TEXT,
            purchase_month          INTEGER,
            shipping_limit_month    INTEGER,
            review_score            REAL,
            payment_type            TEXT,
            customer_state          TEXT,
            seller_state            TEXT,
            price_normalized        REAL,
            freight_value_normalized REAL,
            FOREIGN KEY (product_id) REFERENCES dim_products(product_id)
        )
    """)

    # 5. Insert Data ke Tabel Dimensi
    df_dim = df_transformed[['product_id','product_category_name',
                              'product_weight_g','product_volume_cm3',
                              'category_encoded']].drop_duplicates(subset='product_id')
    df_dim.to_sql('dim_products', conn, if_exists='replace', index=False)
    record_load_log("Load dim_products", f"Berhasil ({len(df_dim)} baris)")

    # 6. Insert Data Kurs ke Tabel Dimensi
    df_rates_dim = df_rates[['date','open_rate','high_rate','low_rate',
                              'close_rate','volume_brl','base_currency','target_currency']]
    df_rates_dim.to_sql('dim_exchange_rates', conn, if_exists='replace', index=False)
    record_load_log("Load dim_exchange_rates", f"Berhasil ({len(df_rates_dim)} baris)")

    # 7. Insert Data ke Tabel Fakta
    fact_cols = ['order_id','product_id','price','freight_value','freight_ratio',
                 'payment_value','payment_value_usd','close_rate','rate_tier',
                 'price_segment','order_status','purchase_month','shipping_limit_month',
                 'review_score','payment_type','customer_state','seller_state',
                 'price_normalized','freight_value_normalized']
    df_fact = df_transformed[[c for c in fact_cols if c in df_transformed.columns]].copy()
    df_fact['price_segment'] = df_fact['price_segment'].astype(str)
    df_fact['rate_tier']     = df_fact['rate_tier'].astype(str)
    df_fact.to_sql('fact_orders', conn, if_exists='replace', index=False)
    record_load_log("Load fact_orders", f"Berhasil ({len(df_fact)} baris)")

    conn.commit()
    total_time = round(time.time() - start_load, 3)
    record_load_log("Total Waktu Load", f"{total_time} detik")
    print(f"Semua tabel berhasil dimuat ke database dalam {total_time}s")

except Exception as e:
    record_load_log("Error", f"{str(e)}")
    print(f"Error: {e}")

display(pd.DataFrame(load_logs))


Semua tabel berhasil dimuat ke database dalam 1.037s


,Langkah,Status,Waktu Eksekusi
0,Load dim_products,Berhasil (32952 baris),2026-06-14 06:56:27
1,Load dim_exchange_rates,Berhasil (35064 baris),2026-06-14 06:56:27
2,Load fact_orders,Berhasil (103200 baris),2026-06-14 06:56:28
3,Total Waktu Load,1.037 detik,2026-06-14 06:56:28


##### **2. 8 Query SQL Analitik**

In [100]:
def run_query(query, title):
    print(f"\n--- {title} ---")
    result = pd.read_sql_query(query, conn)
    display(result)

# 1. Total Transaksi dan Pendapatan
run_query("SELECT COUNT(order_id) as total_order, ROUND(SUM(payment_value),2) as total_revenue_brl, ROUND(SUM(payment_value_usd),2) as total_revenue_usd FROM fact_orders",
          "Total Order dan Revenue (BRL & USD)")

# 2. Top 5 Kategori Produk dengan Penjualan Tertinggi
run_query("""
    SELECT d.product_category_name, COUNT(f.order_id) as sales_count,
           ROUND(SUM(f.payment_value_usd),2) as revenue_usd
    FROM fact_orders f
    JOIN dim_products d ON f.product_id = d.product_id
    GROUP BY 1 ORDER BY 2 DESC LIMIT 5
""", "Top 5 Kategori Produk")

# 3. Analisis Segmen Harga
run_query("SELECT price_segment, COUNT(*) as jumlah, ROUND(AVG(payment_value_usd),2) as avg_usd FROM fact_orders GROUP BY 1",
          "Distribusi Segmen Harga")

# 4. Rata-rata Ongkir per Kategori
run_query("""
    SELECT d.product_category_name, ROUND(AVG(f.freight_value),2) as avg_shipping
    FROM fact_orders f
    JOIN dim_products d ON f.product_id = d.product_id
    GROUP BY 1 ORDER BY 2 DESC LIMIT 5
""", "Rata-rata Ongkir per Kategori")

# 5. Distribusi Metode Pembayaran
run_query("SELECT payment_type, COUNT(*) as jumlah, ROUND(AVG(payment_value_usd),2) as avg_usd FROM fact_orders GROUP BY 1 ORDER BY 2 DESC",
          "Distribusi Metode Pembayaran")

# 6. Analisis Kurs — Revenue per Rate Tier
run_query("SELECT rate_tier, COUNT(*) as jumlah_order, ROUND(AVG(close_rate),4) as avg_kurs, ROUND(SUM(payment_value_usd),2) as total_usd FROM fact_orders GROUP BY 1 ORDER BY 3 DESC",
          "Revenue per Kondisi Kurs BRL/USD")

# 7. Top 5 Negara Bagian Pelanggan
run_query("SELECT customer_state, COUNT(*) as total_order, ROUND(AVG(payment_value_usd),2) as avg_spend_usd FROM fact_orders GROUP BY 1 ORDER BY 2 DESC LIMIT 5",
          "Top 5 Negara Bagian Pelanggan")

# 8. Kurs Tertinggi & Terendah dari Data JSON
run_query("""
    SELECT date, close_rate,
           CASE WHEN close_rate = (SELECT MAX(close_rate) FROM dim_exchange_rates) THEN 'Tertinggi'
                WHEN close_rate = (SELECT MIN(close_rate) FROM dim_exchange_rates) THEN 'Terendah'
           END as keterangan
    FROM dim_exchange_rates
    WHERE close_rate = (SELECT MAX(close_rate) FROM dim_exchange_rates)
       OR close_rate = (SELECT MIN(close_rate) FROM dim_exchange_rates)
""", "Kurs BRL/USD Tertinggi & Terendah")



--- Total Order dan Revenue (BRL & USD) ---


,total_order,total_revenue_brl,total_revenue_usd
0,103200,16995899.32,5017059.67



--- Top 5 Kategori Produk ---


,product_category_name,sales_count,revenue_usd
0,cama_mesa_banho,10160,431276.88
1,beleza_saude,9022,434595.21
2,esporte_lazer,7858,358045.89
3,informatica_acessorios,6887,341830.65
4,moveis_decoracao,6781,302827.96



--- Distribusi Segmen Harga ---


...

,rate_tier,jumlah_order,avg_kurs,total_usd
0,Strong BRL,61314,0.3103,3076928.91
1,Normal BRL,22156,0.2892,1087879.93
2,Weak BRL,19730,0.2565,852250.83



--- Top 5 Negara Bagian Pelanggan ---


,customer_state,total_order,avg_spend_usd
0,SP,43423,43.28
1,RJ,13335,50.62
2,MG,12077,48.54
3,RS,5671,49.28
4,PR,5204,48.73



--- Kurs BRL/USD Tertinggi & Terendah ---


,date,close_rate,keterangan
0,2017-02-08,0.325229,Tertinggi
1,2018-09-30,0.242495,Terendah


##### **3. Unduh File Database**

In [101]:
from google.colab import files
conn.close()
files.download('olist_data_warehouse.db')
print("Silakan simpan file .db ini untuk dikumpulkan bersama laporan.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Silakan simpan file .db ini untuk dikumpulkan bersama laporan.


## **Bagian 2. Pipline ELT (Extract - Load - Transform)**

### **2.1 Extract dan Load**

#### 1. Inisialisasi Database ELT & Metadata Logger

Langkah pertama adalah menyiapkan "wadah" berupa database SQLite baru dan fungsi untuk mencatat riwayat pemuatan data (metadata).

In [102]:
import pandas as pd
import sqlite3
import os
import time

# Nama database khusus untuk ELT
db_elt = "olist_elt_warehouse.db"
conn_elt = sqlite3.connect(db_elt)
elt_metadata_logs = []

def get_file_info(file_path):
    """Fungsi pembantu untuk mengambil ukuran file dalam KB"""
    size_kb = round(os.path.getsize(file_path) / 1024, 2)
    return size_kb

print("Database ELT siap. Logger diinisialisasi.")


Database ELT siap. Logger diinisialisasi.


##### Penjelasan :  membuat database berbeda(olist_elt_warehouse.db) untuk memisahkan hasil kerja ETL sebelumnya dengan ELT ini. Hal ini bertujuan agar data mentah (Raw) tidak tercampur dengan data yang sudah bersih (Gold).

#### 2. ELT - Extract (Mengambil Data Tanpa Preprocessing)

Pada tahap ELT, kita mengambil data apa adanya sesuai dengan yang ada di sumber.

In [103]:
print("Menjalankan ELT: Tahap Extract...")

import urllib.request, json
import pandas as pd

# Sumber 1: CSV dari Google Drive
df_raw_orders = pd.read_csv(
    'https://drive.google.com/uc?export=download&id=1ju7BogZ9kOrFFSSanZZt7pgxYh13aemZ',
    low_memory=False
)

# Sumber 2: JSON dari Google Drive
urllib.request.urlretrieve(
    'https://drive.google.com/uc?export=download&id=1_ciD4VwgfG4HJezwFvaq5VIMwv0O-Llj',
    'brl_usd_hourly.json'
)
with open('brl_usd_hourly.json') as f:
    raw_json_elt = json.load(f)

# Handle berbagai struktur JSON
if isinstance(raw_json_elt, list):
    df_raw_rates = pd.DataFrame(raw_json_elt)
elif isinstance(raw_json_elt, dict):
    data_key = next((k for k in raw_json_elt if isinstance(raw_json_elt[k], list)), None)
    if data_key:
        df_raw_rates = pd.DataFrame(raw_json_elt[data_key])
    else:
        df_raw_rates = pd.DataFrame([raw_json_elt])

print(f"Data Orders : {df_raw_orders.shape[0]} baris berhasil dibaca.")
print(f"Data Kurs   : {df_raw_rates.shape[0]} baris berhasil dibaca.")
print(f"Kolom JSON  : {list(df_raw_rates.columns)}")

Menjalankan ELT: Tahap Extract...
Data Orders : 113425 baris berhasil dibaca.
Data Kurs   : 35064 baris berhasil dibaca.
Kolom JSON  : ['date', 'hour', 'datetime', 'base_currency', 'target_currency', 'open_rate', 'high_rate', 'low_rate', 'close_rate', 'volume_brl', 'source']


#### 3. ELT - Load (Memasukkan ke Tabel Raw)

Data yang sudah dibaca langsung dimasukkan ke dalam tabel database.

In [104]:
print("Menjalankan ELT: Tahap Load ke Tabel Raw...")

start_time = time.time()

# Memuat data langsung ke tabel database dengan awalan 'raw_'
df_raw_orders.to_sql('raw_orders', conn_elt, if_exists='replace', index=False)
df_raw_rates.to_sql('raw_exchange_rates', conn_elt, if_exists='replace', index=False)

execution_time = round(time.time() - start_time, 4)
print(f"Load selesai dalam {execution_time} detik.")


Menjalankan ELT: Tahap Load ke Tabel Raw...
Load selesai dalam 2.1519 detik.


##### Penjelasan : Data dimuat ke tabel raw_orders dan raw_exchange_rates. Perhatikan bahwa kita menggunakan if_exists='replace' untuk memastikan tabel selalu segar jika script dijalankan ulang.

#### 4. Dokumentasi Metadata Proses Load

Tahap ini wajib ada sesuai instruksi untuk mencatat ukuran dan struktur data sebagai bentuk audit.

In [105]:
# Mencatat metadata untuk tabel raw_orders
elt_metadata_logs.append({
    "Nama Tabel"        : "raw_orders",
    "Sumber File"       : "olist_ecommerce_dataset.csv (Google Drive)",
    "Jumlah Kolom"      : df_raw_orders.shape[1],
    "Jumlah Baris"      : df_raw_orders.shape[0],
    "Waktu Load (Detik)": execution_time
})

# Mencatat metadata untuk tabel raw_exchange_rates
elt_metadata_logs.append({
    "Nama Tabel"        : "raw_exchange_rates",
    "Sumber File"       : "brl_usd_hourly.json (Google Drive)",
    "Jumlah Kolom"      : df_raw_rates.shape[1],
    "Jumlah Baris"      : df_raw_rates.shape[0],
    "Waktu Load (Detik)": execution_time
})

print("=" * 85)
print("DOKUMENTASI METADATA PROSES LOAD (ELT RAW)")
print("=" * 85)
df_metadata = pd.DataFrame(elt_metadata_logs)
display(df_metadata)


DOKUMENTASI METADATA PROSES LOAD (ELT RAW)


,Nama Tabel,Sumber File,Jumlah Kolom,Jumlah Baris,Waktu Load (Detik)
0,raw_orders,olist_ecommerce_dataset.csv (Google Drive),42,113425,2.1519
1,raw_exchange_rates,brl_usd_hourly.json (Google Drive),11,35064,2.1519


#### Kesimpulan dari Extract dan Load

1. Extract & Load: Anda telah berhasil memindahkan data dari file flat (CSV/JSON) ke dalam lingkungan database SQL tanpa mengubah isinya.

2. Struktur Data: Metadata menunjukkan bahwa data tetap utuh (raw_orders memiliki 113.425 baris, raw_exchange_rates memiliki 35.064 baris).

3. Kesiapan Transformasi: Sekarang data sudah berada di dalam SQL. Tahap transformasi berikutnya tidak akan lagi menggunakan Python Pandas, melainkan menggunakan perintah SQL Query langsung di dalam database.

### **2.2 Transform**

Pada tahap Transform dalam pipeline ELT, perbedaan utamanya adalah seluruh proses pengolahan data terjadi di dalam database menggunakan mesin SQL, bukan di memori Python (Pandas) seperti pada ETL sebelumnya. Kita akan menggunakan tabel raw_orders dan raw_exchange_rates yang sudah dimuat di database untuk menciptakan tabel baru yang sudah tertransformasi.

#### 1. SQL-Based Data Cleaning & Integration

Menggabungkan data dan membersihkannya (menangani null dan duplikat) langsung melalui query SQL. Pendekatan ini lebih efisien untuk dataset besar karena memanfaatkan optimasi mesin database.

In [106]:
print("Menjalankan ELT Transform: SQL-Based Cleaning & Join...")

cursor_elt = conn_elt.cursor()

sql_transform_query = """
CREATE TABLE transformed_analytics AS
SELECT
    DISTINCT
    o.order_id,
    o.product_id,
    COALESCE(o.price, 0) as price,
    COALESCE(o.freight_value, 0) as freight_value,
    o.product_category_name,
    COALESCE(o.product_weight_g,
        (SELECT AVG(product_weight_g) FROM raw_orders)) as product_weight_g,
    o.payment_value,
    o.payment_type,
    o.order_status,
    o.customer_state,
    o.seller_state,
    o.review_score,
    substr(o.order_purchase_timestamp, 1, 10) as order_date,
    r.close_rate,
    r.open_rate,
    r.high_rate,
    r.low_rate,
    COALESCE(r.volume_brl, 0) as volume_brl,
    ROUND(o.payment_value * COALESCE(r.close_rate, 0), 2) as payment_value_usd
FROM raw_orders o
LEFT JOIN (
    SELECT date,
           AVG(open_rate)  as open_rate,
           MAX(high_rate)  as high_rate,
           MIN(low_rate)   as low_rate,
           AVG(close_rate) as close_rate,
           AVG(volume_brl) as volume_brl
    FROM raw_exchange_rates GROUP BY date
) r ON substr(o.order_purchase_timestamp, 1, 10) = r.date
WHERE o.order_id IS NOT NULL
"""

try:
    cursor_elt.execute("DROP TABLE IF EXISTS transformed_analytics")
    cursor_elt.execute(sql_transform_query)
    conn_elt.commit()
    count = pd.read_sql("SELECT COUNT(*) as n FROM transformed_analytics", conn_elt).iloc[0,0]
    print(f"Cleaning dan Join di sisi Warehouse berhasil. ({count:,} baris)")
except Exception as e:
    print(f"Error Transform SQL: {e}")


Menjalankan ELT Transform: SQL-Based Cleaning & Join...
Cleaning dan Join di sisi Warehouse berhasil. (103,200 baris)


#### 2. SQL-Based Agregasi & Metrik Analitik

Langkah ini memenuhi syarat untuk melakukan agregasi di sisi warehouse. Kita akan membuat tabel ringkasan performa kategori produk.

In [107]:
print("Menjalankan ELT Transform: Warehouse Aggregation...")

sql_aggregation_query = """
CREATE TABLE category_performance_metrics AS
SELECT
    product_category_name,
    COUNT(order_id)                      as total_transactions,
    ROUND(SUM(price), 2)                 as total_revenue,
    ROUND(AVG(price), 2)                 as avg_order_value,
    ROUND(AVG(freight_value), 2)         as avg_shipping_cost,
    ROUND(SUM(payment_value_usd), 2)     as total_revenue_usd
FROM transformed_analytics
GROUP BY product_category_name
ORDER BY total_revenue DESC;
"""

cursor_elt.execute("DROP TABLE IF EXISTS category_performance_metrics")
cursor_elt.execute(sql_aggregation_query)
conn_elt.commit()

print("Agregasi metrik analitik berhasil dibuat.")
display(pd.read_sql("SELECT * FROM category_performance_metrics LIMIT 5", conn_elt))


Menjalankan ELT Transform: Warehouse Aggregation...
Agregasi metrik analitik berhasil dibuat.


,product_category_name,total_transactions,total_revenue,avg_order_value,avg_shipping_cost,total_revenue_usd
0,beleza_saude,9022,1212149.92,134.35,19.13,434560.02
1,relogios_presentes,5799,1187533.96,204.78,16.81,395463.12
2,cama_mesa_banho,10160,963519.89,94.83,18.52,431276.88
3,esporte_lazer,7858,929271.47,118.26,19.75,358045.89
4,informatica_acessorios,6887,806515.13,117.11,18.90,341830.65


#### 3. SQL-Based Feature Engineering

Melakukan feature engineering berbasis query. Membuat fitur baru menggunakan logika CASE dan perhitungan SQL tanpa keluar dari database.

In [108]:
print("Menjalankan ELT Transform: SQL Feature Engineering...")

sql_feature_eng_query = """
CREATE TABLE final_elt_features AS
SELECT
    *,
    (price + freight_value) as gross_transaction_value,
    ROUND(freight_value * 1.0 / NULLIF(price, 0), 4) as freight_ratio,
    CASE
        WHEN price > 200 THEN 'Premium'
        WHEN price > 50  THEN 'Standard'
        ELSE 'Budget'
    END as price_tier,
    CASE
        WHEN product_weight_g > 5000 THEN 'Heavy/Bulky'
        ELSE 'Lightweight'
    END as logistics_type,
    CASE
        WHEN close_rate >= 0.30 THEN 'Strong BRL'
        WHEN close_rate >= 0.27 THEN 'Normal BRL'
        ELSE 'Weak BRL'
    END as rate_tier
FROM transformed_analytics;
"""

cursor_elt.execute("DROP TABLE IF EXISTS final_elt_features")
cursor_elt.execute(sql_feature_eng_query)
conn_elt.commit()

print("Feature Engineering berbasis SQL berhasil.")
display(pd.read_sql("SELECT * FROM final_elt_features LIMIT 5", conn_elt))


Menjalankan ELT Transform: SQL Feature Engineering...
Feature Engineering berbasis SQL berhasil.


,order_id,product_id,price,freight_value,product_category_name,product_weight_g,payment_value,payment_type,order_status,customer_state,...,open_rate,high_rate,low_rate,volume_brl,payment_value_usd,gross_transaction_value,freight_ratio,price_tier,logistics_type,rate_tier
0,e481f51cbdc54678b7cc49136f2d6af7,87285b34884572647811a353c7ac498a,29.99,8.72,utilidades_domesticas,500.0,38.71,credit_card,delivered,SP,...,0.314843,0.319336,0.312721,3.006665e+07,12.19,38.71,0.2908,Budget,Lightweight,Strong BRL
1,53cdb2fc8bc7dce0b6741e2150273451,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,perfumaria,400.0,141.46,boleto,delivered,BA,...,0.259662,0.260985,0.258055,2.699925e+07,36.73,141.46,0.1917,Standard,Lightweight,Weak BRL
2,47770eb9100c2d0c44946d9cf07ec65d,aa4383b373c6aca5d8797843e5594415,159.90,19.22,automotivo,420.0,179.12,credit_card,delivered,GO,...,0.248436,0.250172,0.246656,2.680703e+07,44.50,179.12,0.1202,Standard,Lightweight,Weak BRL
3,949d5b44dbf5de918fe9c16f97b45f8a,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,pet_shop,450.0,72.20,credit_card,delivered,RN,...,0.305671,0.307771,0.303479,2.441279e+07,22.07,72.20,0.6044,Budget,Lightweight,Strong BRL
4,ad21c59c0840e6cb83a9ceb5573f8159,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,papelaria,250.0,28.62,credit_card,delivered,SP,...,0.304660,0.306674,0.302384,2.997216e+07,8.72,28.62,0.4382,Budget,Lightweight,Strong BRL


#### Perbedaan Pendekatan ELT vs ETL

* Tempat Pemrosesan: Pada ETL, pembersihan dilakukan di Python (Pandas) sebelum data masuk database. Pada ELT, data masuk dulu ke tabel raw, baru diproses menggunakan SQL DDL/DML (CREATE TABLE AS SELECT).

* Handling Null: Pada ELT, kita menggunakan fungsi SQL COALESCE dan subquery untuk mengisi nilai kosong (seperti rata-rata berat produk), yang menunjukkan logika transformasi di level database.

* Efisiensi: ELT memungkinkan kita menyimpan history data mentah di tabel raw sehingga jika logika transformasi berubah, kita tidak perlu mengekstrak ulang dari sumber asli.

## **Bagian 3. Dashboard Analitik**

Dashboard dibangun menggunakan **Power BI Desktop** yang terhubung langsung ke **PostgreSQL Data Warehouse** hasil pipeline ETL dan ELT. Koneksi menggunakan mode **DirectQuery** sehingga data di Power BI selalu real-time sesuai isi warehouse.
- ✅ Key Performance Indicators (KPI)
- ✅ Analisis tren berbasis waktu
- ✅ Distribusi dan perbandingan data
- ✅ Filter interaktif untuk eksplorasi data

### **3.1 Setup Koneksi Neon PostgreSQL Warehouse**

In [109]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

PG_HOST = "ep-shiny-morning-aok0mute-pooler.c-2.ap-southeast-1.aws.neon.tech"
PG_PORT = "5432"
PG_USER = "neondb_owner"
PG_PASSWORD = "npg_EIlvTr5z2BZX"
PG_DB = "neondb"

CONNECTION_STRING = (
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}"
    f"@{PG_HOST}:{PG_PORT}/{PG_DB}"
    "?sslmode=require"
)

try:
    engine_pg = create_engine(CONNECTION_STRING)

    with engine_pg.connect() as conn:
        version = conn.execute(
            text("SELECT version();")
        ).fetchone()[0]

    print("=" * 65)
    print("KONEKSI NEON POSTGRESQL BERHASIL")
    print("=" * 65)
    print(f"Host     : {PG_HOST}")
    print(f"Database : {PG_DB}")
    print(f"User     : {PG_USER}")
    print(f"Version  : {version}")
    print("=" * 65)

except Exception as e:
    print(f"[ERROR] Koneksi Neon gagal: {e}")

KONEKSI NEON POSTGRESQL BERHASIL
Host     : ep-shiny-morning-aok0mute-pooler.c-2.ap-southeast-1.aws.neon.tech
Database : neondb
User     : neondb_owner
Version  : PostgreSQL 18.4 (6e15e70) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


### **3.2 ETL — Load ke PostgreSQL (schema: public)**

Pipeline ETL memproses data di Python/Pandas terlebih dahulu, lalu hasilnya dimuat ke PostgreSQL sebagai **Star Schema** (fact + dimensi).

**Tabel yang dibuat:**
- `public.fact_orders` — tabel fakta transaksi
- `public.dim_products` — dimensi produk  
- `public.dim_customers` — dimensi pelanggan
- `public.dim_exchange_rates` — dimensi kurs BRL/USD
- `public.vw_kpi_summary` — view KPI siap pakai Power BI
- `public.vw_monthly_revenue` — view tren bulanan
- `public.vw_category_performance` — view performa kategori
- `public.vw_customer_state` — view distribusi negara bagian

In [110]:
import time
from sqlalchemy import text

print("=" * 65)
print(" ETL LOAD → NEON POSTGRESQL [schema: public]")
print("=" * 65)

start_total = time.time()

try:

    with engine_pg.begin() as conn:

        conn.execute(text("""
            DROP VIEW IF EXISTS public.vw_kpi_summary CASCADE
        """))

        conn.execute(text("""
            DROP VIEW IF EXISTS public.vw_monthly_revenue CASCADE
        """))

        conn.execute(text("""
            DROP VIEW IF EXISTS public.vw_customer_state CASCADE
        """))

        conn.execute(text("""
            DROP VIEW IF EXISTS public.vw_payment_distribution CASCADE
        """))

    print("✓ View lama dibersihkan")

    print("\n[1/4] Loading dim_products...")

    dim_products_pg = (
        df_transformed[
            [c for c in [
                'product_id',
                'product_category_name',
                'product_weight_g',
                'product_length_cm',
                'product_height_cm',
                'product_width_cm'
            ] if c in df_transformed.columns]
        ]
        .drop_duplicates(subset=['product_id'])
        .reset_index(drop=True)
    )

    dim_products_pg.to_sql(
        "dim_products",
        engine_pg,
        schema="public",
        if_exists="replace",
        index=False
    )

    print(f"✓ {len(dim_products_pg):,} rows")

    print("[2/4] Loading dim_customers...")

    dim_customers_pg = (
        df_transformed[
            [c for c in [
                'customer_id',
                'customer_unique_id',
                'customer_state',
                'customer_city'
            ] if c in df_transformed.columns]
        ]
        .drop_duplicates(subset=['customer_id'])
        .reset_index(drop=True)
    )

    dim_customers_pg.to_sql(
        "dim_customers",
        engine_pg,
        schema="public",
        if_exists="replace",
        index=False
    )

    print(f"✓ {len(dim_customers_pg):,} rows")

    print("[3/4] Loading dim_exchange_rates...")

    dim_rates_pg = df_rates_raw.copy()

    dim_rates_pg.columns = [
        str(c).lower().replace(" ", "_")
        for c in dim_rates_pg.columns
    ]

    dim_rates_pg.to_sql(
        "dim_exchange_rates",
        engine_pg,
        schema="public",
        if_exists="replace",
        index=False
    )

    print(f"✓ {len(dim_rates_pg):,} rows")

    print("[4/4] Loading fact_orders...")

    fact_orders_pg = (
        df_transformed[
            [c for c in [
                'order_id',
                'product_id',
                'customer_id',
                'order_date',
                'price',
                'freight_value',
                'payment_value',
                'payment_type',
                'order_status',
                'customer_state',
                'seller_state',
                'review_score',
                'close_rate',
                'payment_value_usd'
            ] if c in df_transformed.columns]
        ]
        .copy()
    )

    fact_orders_pg.to_sql(
        "fact_orders",
        engine_pg,
        schema="public",
        if_exists="replace",
        index=False,
        chunksize=500
    )

    print(f"✓ {len(fact_orders_pg):,} rows")

    print("\nCreating ETL Views...")

    with engine_pg.begin() as conn:

        conn.execute(text("""
        CREATE OR REPLACE VIEW public.vw_kpi_summary AS
        SELECT
            COUNT(*) AS total_transaksi,
            ROUND(SUM(price)::numeric,2) AS total_revenue_brl,
            ROUND(SUM(payment_value_usd)::numeric,2) AS total_revenue_usd,
            ROUND(AVG(price)::numeric,2) AS avg_harga_brl,
            ROUND(AVG(freight_value)::numeric,2) AS avg_ongkir_brl,
            ROUND(AVG(review_score)::numeric,2) AS avg_review_score
        FROM public.fact_orders
        """))

        conn.execute(text("""
        CREATE OR REPLACE VIEW public.vw_monthly_revenue AS
        SELECT
            TO_CHAR(order_date::date,'YYYY-MM') AS bulan,
            COUNT(*) AS total_order,
            ROUND(SUM(price)::numeric,2) AS revenue_brl,
            ROUND(SUM(payment_value_usd)::numeric,2) AS revenue_usd
        FROM public.fact_orders
        GROUP BY bulan
        ORDER BY bulan
        """))

        conn.execute(text("""
        CREATE OR REPLACE VIEW public.vw_customer_state AS
        SELECT
            customer_state,
            COUNT(*) AS total_order,
            ROUND(SUM(payment_value_usd)::numeric,2) AS total_spend_usd
        FROM public.fact_orders
        GROUP BY customer_state
        ORDER BY total_order DESC
        """))

        conn.execute(text("""
        CREATE OR REPLACE VIEW public.vw_payment_distribution AS
        SELECT
            payment_type,
            COUNT(*) AS jumlah,
            ROUND(SUM(payment_value_usd)::numeric,2) AS total_usd
        FROM public.fact_orders
        GROUP BY payment_type
        """))

    print("✓ VIEW vw_kpi_summary")
    print("✓ VIEW vw_monthly_revenue")
    print("✓ VIEW vw_customer_state")
    print("✓ VIEW vw_payment_distribution")

    total_time = round(time.time() - start_total, 2)

    print("\n" + "=" * 65)
    print(f"ETL LOAD SELESAI ({total_time} detik)")
    print("=" * 65)

except Exception as e:
    print(f"[ERROR] ETL Load gagal: {e}")

 ETL LOAD → NEON POSTGRESQL [schema: public]
✓ View lama dibersihkan

[1/4] Loading dim_products...
✓ 32,952 rows
[2/4] Loading dim_customers...
✓ 99,441 rows
[3/4] Loading dim_exchange_rates...
✓ 35,064 rows
[4/4] Loading fact_orders...
✓ 103,200 rows

Creating ETL Views...
✓ VIEW vw_kpi_summary
✓ VIEW vw_monthly_revenue
✓ VIEW vw_customer_state
✓ VIEW vw_payment_distribution

ETL LOAD SELESAI (116.8 detik)


In [111]:
df_transformed.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,volume_brl,price_normalized,freight_value_normalized,category_encoded,purchase_month,shipping_limit_month,freight_ratio,product_volume_cm3,price_segment,rate_tier
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.006665e+07,0.106234,0.021285,73,10,10.0,0.290764,1976.0,Low,Strong BRL
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,2.699925e+07,0.429639,0.055556,63,7,7.0,0.191744,4693.0,Medium,Weak BRL
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,2.680703e+07,0.579840,0.046915,9,8,8.0,0.120200,9576.0,Medium,Weak BRL
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,2.441279e+07,0.160955,0.066393,64,11,11.0,0.604444,6000.0,Low,Strong BRL
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,2.997216e+07,0.069450,0.021285,60,2,2.0,0.438191,11475.0,Low,Strong BRL


### **3.3 ELT — Load RAW + Transform SQL ke PostgreSQL (schema: elt)**

Pipeline ELT memuat data **mentah** langsung ke PostgreSQL, lalu seluruh transformasi dilakukan menggunakan **SQL di dalam warehouse** (bukan Python).

**Tabel yang dibuat:**
- `elt.raw_orders` — data mentah transaksi (tanpa preprocessing)
- `elt.raw_exchange_rates` — data mentah kurs (tanpa preprocessing)
- `elt.transformed_analytics` — hasil cleaning & join via SQL
- `elt.final_elt_features` — hasil feature engineering via SQL
- `elt.category_performance_metrics` — agregasi kategori via SQL
- `elt.vw_elt_kpi` — view KPI untuk Power BI
- `elt.vw_elt_monthly` — view tren bulanan untuk Power BI

In [112]:
import pandas as pd
import urllib.request
import json as json_lib
import time
from sqlalchemy import text

print("=" * 65)
print("  ELT EXTRACT & LOAD RAW → POSTGRESQL [schema: elt]")
print("=" * 65)

elt_load_logs = []

try:
    # Buat schema elt jika belum ada
    with engine_pg.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS elt"))
        conn.commit()
    print("\n[SCHEMA] Schema 'elt' siap")

    # --- EXTRACT (tanpa preprocessing) ---
    print("\n[EXTRACT] Membaca data mentah dari sumber...")
    ID_FILE_CSV  = '1ju7BogZ9kOrFFSSanZZt7pgxYh13aemZ'
    ID_FILE_JSON = '1_ciD4VwgfG4HJezwFvaq5VIMwv0O-Llj'

    start = time.time()
    df_raw_orders_elt = pd.read_csv(
        f'https://drive.google.com/uc?export=download&id={ID_FILE_CSV}',
        low_memory=False
    )
    print(f"     ✓ raw_orders: {df_raw_orders_elt.shape[0]:,} baris × {df_raw_orders_elt.shape[1]} kolom")

    urllib.request.urlretrieve(
        f'https://drive.google.com/uc?export=download&id={ID_FILE_JSON}',
        'brl_usd_hourly.json'
    )
    with open('brl_usd_hourly.json') as f:
        raw_json_elt = json_lib.load(f)

    if isinstance(raw_json_elt, list):
        df_raw_rates_elt = pd.DataFrame(raw_json_elt)
    elif isinstance(raw_json_elt, dict):
        data_key = next((k for k in raw_json_elt if isinstance(raw_json_elt[k], list)), None)
        df_raw_rates_elt = pd.DataFrame(raw_json_elt[data_key]) if data_key else pd.DataFrame([raw_json_elt])
    print(f"     ✓ raw_exchange_rates: {df_raw_rates_elt.shape[0]:,} baris × {df_raw_rates_elt.shape[1]} kolom")

    # --- LOAD RAW (langsung ke PostgreSQL tanpa transform) ---
    print("\n[LOAD RAW] Memuat data mentah ke PostgreSQL...")
    df_raw_orders_elt.to_sql('raw_orders', engine_pg, schema='elt',
                              if_exists='replace', index=False, method='multi', chunksize=1000)
    elt_load_logs.append({"Tabel": "elt.raw_orders", "Baris": len(df_raw_orders_elt),
                           "Sumber": "CSV Google Drive", "Status": "✓ Berhasil"})
    print(f"     ✓ elt.raw_orders ({len(df_raw_orders_elt):,} baris)")

    df_raw_rates_elt.to_sql('raw_exchange_rates', engine_pg, schema='elt',
                             if_exists='replace', index=False, method='multi', chunksize=1000)
    elt_load_logs.append({"Tabel": "elt.raw_exchange_rates", "Baris": len(df_raw_rates_elt),
                           "Sumber": "JSON Google Drive", "Status": "✓ Berhasil"})
    print(f"     ✓ elt.raw_exchange_rates ({len(df_raw_rates_elt):,} baris)")

    load_time = round(time.time() - start, 2)
    print(f"\n  Load RAW selesai dalam {load_time} detik")
    print("\n[METADATA] Dokumentasi Proses Load:")
    display(pd.DataFrame(elt_load_logs))

except Exception as e:
    print(f"[ERROR] ELT Extract/Load gagal: {e}")

  ELT EXTRACT & LOAD RAW → POSTGRESQL [schema: elt]

[SCHEMA] Schema 'elt' siap

[EXTRACT] Membaca data mentah dari sumber...
     ✓ raw_orders: 113,425 baris × 42 kolom
     ✓ raw_exchange_rates: 35,064 baris × 11 kolom

[LOAD RAW] Memuat data mentah ke PostgreSQL...
     ✓ elt.raw_orders (113,425 baris)
     ✓ elt.raw_exchange_rates (35,064 baris)

  Load RAW selesai dalam 210.01 detik

[METADATA] Dokumentasi Proses Load:


,Tabel,Baris,Sumber,Status
0,elt.raw_orders,113425,CSV Google Drive,✓ Berhasil
1,elt.raw_exchange_rates,35064,JSON Google Drive,✓ Berhasil


In [113]:
print("=" * 65)
print("  ELT TRANSFORM → SQL di dalam PostgreSQL [schema: elt]")
print("=" * 65)

try:
    with engine_pg.connect() as conn:

        # --- TABEL 1: transformed_analytics (Cleaning + Join via SQL) ---
        print("\n[SQL 1/3] Membuat elt.transformed_analytics (Cleaning + Join)...")
        conn.execute(text("DROP TABLE IF EXISTS elt.transformed_analytics"))
        conn.execute(text("""
            CREATE TABLE elt.transformed_analytics AS
            SELECT DISTINCT
                o.order_id,
                o.product_id,
                o.customer_id,
                COALESCE(NULLIF(CAST(o.price AS TEXT), ''), '0')::float          AS price,
                COALESCE(NULLIF(CAST(o.freight_value AS TEXT), ''), '0')::float  AS freight_value,
                COALESCE(o.product_category_name, 'Unknown')                     AS product_category_name,
                COALESCE(
                    NULLIF(CAST(o.product_weight_g AS TEXT), '')::float,
                    (SELECT AVG(NULLIF(CAST(product_weight_g AS TEXT),'')::float) FROM elt.raw_orders)
                )                                                                 AS product_weight_g,
                COALESCE(NULLIF(CAST(o.payment_value AS TEXT), ''), '0')::float  AS payment_value,
                COALESCE(o.payment_type, 'Unknown')                              AS payment_type,
                COALESCE(o.order_status, 'Unknown')                              AS order_status,
                COALESCE(o.customer_state, 'Unknown')                            AS customer_state,
                COALESCE(o.seller_state, 'Unknown')                              AS seller_state,
                COALESCE(NULLIF(CAST(o.review_score AS TEXT), ''), '0')::float   AS review_score,
                SUBSTRING(CAST(o.order_purchase_timestamp AS TEXT), 1, 10)       AS order_date,
                r.close_rate,
                r.open_rate,
                r.high_rate,
                r.low_rate,
                COALESCE(r.volume_brl, 0)                                        AS volume_brl,
               CAST(
                    ROUND(
                        (
                            COALESCE(NULLIF(CAST(o.payment_value AS TEXT),'')::numeric, 0)
                            *
                            COALESCE(r.close_rate::numeric, 0)
                        ),
                        2
                    )
                AS NUMERIC(18,2)) AS payment_value_usd
            FROM elt.raw_orders o
            LEFT JOIN (
                SELECT date,
                       AVG(open_rate)  AS open_rate,
                       MAX(high_rate)  AS high_rate,
                       MIN(low_rate)   AS low_rate,
                       AVG(close_rate) AS close_rate,
                       AVG(volume_brl) AS volume_brl
                FROM elt.raw_exchange_rates
                GROUP BY date
            ) r ON SUBSTRING(CAST(o.order_purchase_timestamp AS TEXT), 1, 10) = r.date
            WHERE o.order_id IS NOT NULL
        """))
        conn.commit()
        n1 = conn.execute(text("SELECT COUNT(*) FROM elt.transformed_analytics")).scalar()
        print(f"     ✓ {n1:,} baris berhasil dibuat")

        # --- TABEL 2: category_performance_metrics (Agregasi via SQL) ---
        print("\n[SQL 2/3] Membuat elt.category_performance_metrics (Agregasi)...")
        conn.execute(text("DROP TABLE IF EXISTS elt.category_performance_metrics"))
        conn.execute(text("""
            CREATE TABLE elt.category_performance_metrics AS
            SELECT
                product_category_name,
                COUNT(order_id)                             AS total_transaksi,
                ROUND(SUM(price)::numeric, 2)               AS total_revenue_brl,
                ROUND(AVG(price)::numeric, 2)               AS avg_order_value_brl,
                ROUND(AVG(freight_value)::numeric, 2)       AS avg_ongkir_brl,
                ROUND(SUM(payment_value_usd)::numeric, 2)   AS total_revenue_usd
            FROM elt.transformed_analytics
            GROUP BY product_category_name
            ORDER BY total_revenue_brl DESC
        """))
        conn.commit()
        print("     ✓ Agregasi kategori selesai")

        # --- TABEL 3: final_elt_features (Feature Engineering via SQL) ---
        print("\n[SQL 3/3] Membuat elt.final_elt_features (Feature Engineering)...")
        conn.execute(text("DROP TABLE IF EXISTS elt.final_elt_features CASCADE"))
        conn.execute(text("""
            CREATE TABLE elt.final_elt_features AS
            SELECT *,
                (price + freight_value)                                  AS gross_transaction_value,
                ROUND((freight_value / NULLIF(price, 0))::numeric, 4)   AS freight_ratio,
                CASE
                    WHEN price > 200 THEN 'Premium'
                    WHEN price > 50  THEN 'Standard'
                    ELSE 'Budget'
                END                                                       AS price_tier,
                CASE
                    WHEN product_weight_g > 5000 THEN 'Heavy/Bulky'
                    ELSE 'Lightweight'
                END                                                       AS logistics_type,
                CASE
                    WHEN close_rate >= 0.30 THEN 'Strong BRL'
                    WHEN close_rate >= 0.27 THEN 'Normal BRL'
                    ELSE 'Weak BRL'
                END                                                       AS rate_tier
            FROM elt.transformed_analytics
        """))
        conn.commit()
        n3 = conn.execute(text("SELECT COUNT(*) FROM elt.final_elt_features")).scalar()
        print(f"     ✓ {n3:,} baris berhasil dibuat")

        # --- VIEW untuk Power BI ---
        print("\nMembuat VIEW analitik ELT di PostgreSQL...")
        views_elt = {
            "vw_elt_kpi": """
                CREATE OR REPLACE VIEW elt.vw_elt_kpi AS
                SELECT
                    COUNT(order_id)                              AS total_transaksi,
                    ROUND(SUM(price)::numeric, 2)               AS total_revenue_brl,
                    ROUND(SUM(payment_value_usd)::numeric, 2)   AS total_revenue_usd,
                    ROUND(AVG(price)::numeric, 2)               AS avg_harga_brl,
                    ROUND(AVG(freight_value)::numeric, 2)       AS avg_ongkir_brl,
                    ROUND(AVG(review_score)::numeric, 2)        AS avg_review
                FROM elt.final_elt_features
            """,
            "vw_elt_monthly": """
                CREATE OR REPLACE VIEW elt.vw_elt_monthly AS
                SELECT
                    TO_CHAR(TO_DATE(order_date,'YYYY-MM-DD'),'YYYY-MM') AS bulan,
                    COUNT(order_id)                              AS total_order,
                    ROUND(SUM(price)::numeric, 2)               AS revenue_brl,
                    ROUND(SUM(payment_value_usd)::numeric, 2)   AS revenue_usd,
                    ROUND(AVG(review_score)::numeric, 2)        AS avg_review
                FROM elt.final_elt_features
                GROUP BY bulan ORDER BY bulan
            """,
            "vw_elt_state": """
                CREATE OR REPLACE VIEW elt.vw_elt_state AS
                SELECT
                    customer_state,
                    COUNT(order_id)                              AS total_order,
                    ROUND(AVG(payment_value_usd)::numeric, 2)   AS avg_spend_usd,
                    ROUND(SUM(payment_value_usd)::numeric, 2)   AS total_spend_usd
                FROM elt.final_elt_features
                GROUP BY customer_state ORDER BY total_order DESC
            """,
            "vw_elt_price_tier": """
                CREATE OR REPLACE VIEW elt.vw_elt_price_tier AS
                SELECT
                    price_tier,
                    COUNT(*)                                     AS jumlah,
                    ROUND(AVG(payment_value_usd)::numeric, 2)   AS avg_usd,
                    ROUND(SUM(payment_value_usd)::numeric, 2)   AS total_usd
                FROM elt.final_elt_features
                GROUP BY price_tier
            """
        }

        for vname, vsql in views_elt.items():
            conn.execute(text(vsql))
            conn.commit()
            print(f"     ✓ VIEW {vname}")

    print(f"\n{'='*65}")
    print("  ELT TRANSFORM SELESAI — Data siap di PostgreSQL!")
    print(f"{'='*65}")

except Exception as e:
    print(f"[ERROR] ELT Transform gagal: {e}")

  ELT TRANSFORM → SQL di dalam PostgreSQL [schema: elt]

[SQL 1/3] Membuat elt.transformed_analytics (Cleaning + Join)...
     ✓ 103,200 baris berhasil dibuat

[SQL 2/3] Membuat elt.category_performance_metrics (Agregasi)...
     ✓ Agregasi kategori selesai

[SQL 3/3] Membuat elt.final_elt_features (Feature Engineering)...
     ✓ 103,200 baris berhasil dibuat

Membuat VIEW analitik ELT di PostgreSQL...
     ✓ VIEW vw_elt_kpi
     ✓ VIEW vw_elt_monthly
     ✓ VIEW vw_elt_state
     ✓ VIEW vw_elt_price_tier

  ELT TRANSFORM SELESAI — Data siap di PostgreSQL!


### **3.4 Verifikasi Warehouse & KPI Summary**

In [114]:
print("=" * 65)
print("  VERIFIKASI TABEL DI POSTGRESQL WAREHOUSE")
print("=" * 65)

try:
    with engine_pg.connect() as conn:

        # Cek semua tabel & view
        tables_check = [
            ("public", "fact_orders"),
            ("public", "dim_products"),
            ("public", "dim_customers"),
            ("public", "dim_exchange_rates"),
            ("elt",    "raw_orders"),
            ("elt",    "raw_exchange_rates"),
            ("elt",    "transformed_analytics"),
            ("elt",    "final_elt_features"),
            ("elt",    "category_performance_metrics"),
        ]

        check_results = []
        for schema, tbl in tables_check:
            try:
                n = conn.execute(text(f'SELECT COUNT(*) FROM {schema}."{tbl}"')).scalar()
                check_results.append({"Schema": schema, "Tabel": tbl, "Jumlah Baris": f"{n:,}", "Status": "✓ Ada"})
            except:
                check_results.append({"Schema": schema, "Tabel": tbl, "Jumlah Baris": "-", "Status": "✗ Tidak Ada"})

        print("\nRingkasan Tabel di Warehouse:")
        display(pd.DataFrame(check_results))

        # KPI Summary dari ETL
        print("\n" + "=" * 65)
        print("  KPI UTAMA — dari ETL Pipeline (public.vw_kpi_summary)")
        print("=" * 65)
        df_kpi_etl = pd.read_sql("SELECT * FROM public.vw_kpi_summary", engine_pg)
        kpi_etl = pd.DataFrame({
            'KPI': ['Total Transaksi', 'Total Revenue (BRL)', 'Total Revenue (USD)',
                    'Rata-rata Harga (BRL)', 'Rata-rata Ongkir (BRL)', 'Rata-rata Review Score'],
            'Nilai (ETL)': [
                f"{df_kpi_etl['total_transaksi'].iloc[0]:,}",
                f"R$ {df_kpi_etl['total_revenue_brl'].iloc[0]:,.2f}",
                f"$ {df_kpi_etl['total_revenue_usd'].iloc[0]:,.2f}",
                f"R$ {df_kpi_etl['avg_harga_brl'].iloc[0]:,.2f}",
                f"R$ {df_kpi_etl['avg_ongkir_brl'].iloc[0]:,.2f}",
                f"{df_kpi_etl['avg_review_score'].iloc[0]:.2f} / 5.0",
            ]
        })

        # KPI Summary dari ELT
        df_kpi_elt = pd.read_sql("SELECT * FROM elt.vw_elt_kpi", engine_pg)
        kpi_etl['Nilai (ELT)'] = [
            f"{df_kpi_elt['total_transaksi'].iloc[0]:,}",
            f"R$ {df_kpi_elt['total_revenue_brl'].iloc[0]:,.2f}",
            f"$ {df_kpi_elt['total_revenue_usd'].iloc[0]:,.2f}",
            f"R$ {df_kpi_elt['avg_harga_brl'].iloc[0]:,.2f}",
            f"R$ {df_kpi_elt['avg_ongkir_brl'].iloc[0]:,.2f}",
            f"{df_kpi_elt['avg_review'].iloc[0]:.2f} / 5.0",
        ]
        display(kpi_etl)

except Exception as e:
    print(f"[ERROR] Verifikasi gagal: {e}")

  VERIFIKASI TABEL DI POSTGRESQL WAREHOUSE

Ringkasan Tabel di Warehouse:


,Schema,Tabel,Jumlah Baris,Status
0,public,fact_orders,"103,200",✓ Ada
1,public,dim_products,"32,952",✓ Ada
2,public,dim_customers,"99,441",✓ Ada
3,public,dim_exchange_rates,"35,064",✓ Ada
4,elt,raw_orders,"113,425",✓ Ada
5,elt,raw_exchange_rates,"35,064",✓ Ada
6,elt,transformed_analytics,"103,200",✓ Ada
7,elt,final_elt_features,"103,200",✓ Ada
8,elt,category_performance_metrics,74,✓ Ada



  KPI UTAMA — dari ETL Pipeline (public.vw_kpi_summary)


,KPI,Nilai (ETL),Nilai (ELT)
0,Total Transaksi,"103,200","103,200"
1,Total Revenue (BRL),"R$ 10,360,193.70","R$ 12,743,924.00"
2,Total Revenue (USD),"$ 5,017,059.67","$ 5,017,024.49"
3,Rata-rata Harga (BRL),R$ 100.39,R$ 123.49
4,Rata-rata Ongkir (BRL),R$ 20.08,R$ 19.96
5,Rata-rata Review Score,4.07 / 5.0,4.03 / 5.0
